# UR5e VLA bed — DAgger round (Kaggle, free)
Inputs: the base dataset **vla-bed-<recipe>** and the **train notebook's output** (the checkpoint that drives). Accelerator GPU T4 x2, Internet ON. The selected checkpoint runs closed-loop on 400 fresh train seeds (seed block 20 000, disjoint from the recipes and the frozen suite) while the capped scripted oracle labels every frame (`sim/vla-bed/dagger.py`); the rollouts are packed as `vla-bed-v7.zip` for the next train session (attach it next to the base dataset; `train.ipynb` merges base ∪ rollouts on the fly). Plan of 6 Sep 2026, step 5.

In [ ]:
# DAgger round on the checkpoint selected from the base recipe's evaluation (R11); one session ≈ 400 × 30 s ≈ 3.5 h on a T4.
RUN = "baseline"
RECIPE = "auto"           # the base recipe = the single vla-bed-<recipe> dataset attached
CHECKPOINT = "selected"   # "selected" = results/p5/*/<RUN>/selected.json from the eval zip if attached, else the last checkpoint; or a step like "005000"
EPISODES = 400
MAX_HOURS = 6.0
SAMPLING_SEED = 0

In [ ]:
import os, subprocess, sys, time, json, pathlib
REPO = "https://github.com/santapong/RoboLLM.git"; BRANCH = "experiment/ur5e-vla-bed"
ROOT = pathlib.Path("/kaggle/working/RoboLLM")
# Kaggle mounts the uploaded zip under /kaggle/input/<slug>/ with or without the zip's top folder; find the manifest.
RECIPE = globals().get("RECIPE", "auto")
# A bundle uploaded as split parts (vla-bed-<recipe>.zip.partNN + SHA256SUMS, made by split -b 9M) is joined and unpacked under /tmp first.
import hashlib, zipfile
for sums in pathlib.Path("/kaggle/input").rglob("SHA256SUMS"):
    parts = sorted(sums.parent.glob("*.zip.part*"))
    if not parts: continue
    name = parts[0].name.split(".zip.part")[0]; joined = pathlib.Path("/tmp/bundles") / f"{name}.zip"; joined.parent.mkdir(parents=True, exist_ok=True)
    if not joined.exists():
        with open(joined, "wb") as out:
            for part in parts: out.write(part.read_bytes())
    want = next((line.split()[0] for line in sums.read_text().splitlines() if line.strip().endswith(f" {name}.zip")), None)
    got = hashlib.sha256(joined.read_bytes()).hexdigest()
    assert want is None or got == want, f"{name}.zip sha256 {got} != {want} (parts incomplete?)"
    dest = pathlib.Path("/tmp/bundles") / name
    if not dest.exists(): zipfile.ZipFile(joined).extractall(dest)
    print("joined", len(parts), "parts →", dest, "sha256 OK" if want else "sha256 unchecked")
roots = [pathlib.Path("/kaggle/input"), pathlib.Path("/tmp/bundles")]
found = {json.load(open(p)).get("recipe"): p.parent for r in roots if r.exists() for p in r.rglob("manifest.json") if (p.parent / "train").is_dir()}
if RECIPE == "auto":   # exactly one bed dataset attached → its recipe
    assert len(found) == 1, "attach exactly one vla-bed-<recipe> dataset or set RECIPE; found: " + str(found)
    RECIPE = next(iter(found))
assert RECIPE in found, f"add the private dataset vla-bed-{RECIPE} to this notebook (Add Input); found: " + str(found)
DATA = found[RECIPE]; print("dataset root", DATA, "recipe", RECIPE)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT)
link = ROOT / "datasets" / "vla-bed" / RECIPE; link.parent.mkdir(parents=True, exist_ok=True)
MANIFEST = f"datasets/vla-bed/{RECIPE}/manifest.json"   # the frozen suite: v3 records the same 100 evaluation seeds/targets as v2
if not link.exists(): link.symlink_to(DATA)          # every default path in the bed now resolves to the uploaded data
MEN = ROOT / "sim" / "vla-bed" / "assets" / "mujoco_menagerie"   # robot models are not vendored (BSD notices in NOTICES.md); pinned sparse clone, as scripts/pi_setup.sh does
if not (MEN / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "--filter=blob:none", "--no-checkout", "https://github.com/google-deepmind/mujoco_menagerie.git", str(MEN)], check=True)
    subprocess.run(["git", "-C", str(MEN), "sparse-checkout", "set", "universal_robots_ur5e", "robotiq_2f85"], check=True)
subprocess.run(["git", "-C", str(MEN), "checkout", "--quiet", "e4049d0a3bfd58d2a3081614e6777d4007e3f86a"], check=True)
print("menagerie", subprocess.run(["git", "-C", str(MEN), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "sim/vla-bed/requirements-record.txt", "mujoco==3.10.0", "pyyaml", "av"], check=True)  # the bed's physics is not in LeRobot's extras
subprocess.run("apt-get install -y -qq libosmesa6 > /dev/null 2>&1 || true", shell=True)   # MuJoCo fallback renderer
os.environ["MUJOCO_GL"] = "egl"; os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "bf16 native", torch.cuda.is_bf16_supported())
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Which renderer works here? EGL (NVIDIA) first, OSMesa second.
import os, subprocess, sys
def try_gl(backend):
    r = subprocess.run([sys.executable, "-c", "import mujoco,numpy as np; m=mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=\"1\"/></worldbody></mujoco>'); d=mujoco.MjData(m); r=mujoco.Renderer(m,64,64); r.update_scene(d); print(r.render().mean())"], env={**os.environ, "MUJOCO_GL": backend}, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-200:]
for b in ("egl", "osmesa"):
    ok, msg = try_gl(b); print(b, "OK" if ok else "FAIL", msg if not ok else "")
    if ok: os.environ["MUJOCO_GL"] = b; break
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
# Find the training output zip under /kaggle/input and unpack it OUTSIDE /kaggle/working — the Output tab would otherwise carry every checkpoint twice (8.24 GB on 4 Sep 2026).
import glob, zipfile, pathlib, shutil, os, json
zips = glob.glob(f"/kaggle/input/**/vla-bed-{RUN}-output" + ("" if RECIPE == "v2" else f"-{RECIPE}") + ".zip", recursive=True)
assert zips, "attach the train notebook's output (Add Input → Notebook output) — found: " + str(glob.glob("/kaggle/input/**/*.zip", recursive=True)[:10])
UNPACK = pathlib.Path("/tmp/vla-bed-unpacked"); shutil.rmtree(UNPACK, ignore_errors=True); shutil.rmtree("/kaggle/working/unpacked", ignore_errors=True)
zipfile.ZipFile(zips[0]).extractall(UNPACK)
CKPTS = {pm.parent.name: str(pm) for pm in sorted(UNPACK.glob(f"artifacts/{RUN}/kaggle/*/pretrained_model")) if pm.parent.name.isdigit()}   # skip LeRobot's `last` alias (a copy of the final step)
print("checkpoints:", sorted(CKPTS))
for rr in sorted(UNPACK.glob("run_record.json")) + sorted(UNPACK.glob("results/p5/*/*/*/nominal.json")):
    d = json.load(open(rr))
    if "steps_done" in d: print("train record:", {k: d.get(k) for k in ("status", "steps_done", "steps_per_s", "wall_s", "gpu", "peak_vram_gb")})
    else:
        lab = d["policy"]["label"]; b = d[lab]; print("quick check", lab, {k: b.get(k) for k in ("n", "success_rate", "ci95_wilson_success", "safety", "progress_mean", "episode_len_mean", "faults")})
for rr in sorted(UNPACK.glob("results/p5/*/*/*/nominal.json")):   # bring the quick-check JSON into this session's results too
    dst_json = pathlib.Path("sim/vla-bed/results/p5") / rr.relative_to(UNPACK / "results" / "p5"); dst_json.parent.mkdir(parents=True, exist_ok=True); shutil.copy(rr, dst_json)


In [ ]:
# Choose the driving checkpoint, run the round, print the policy's own success on the fresh seeds (a free extra evaluation, unpaired).
import glob, json, subprocess, sys, time, pathlib
sel = sorted(glob.glob("/kaggle/input/**/results/p5/*/" + RUN + "/selected.json", recursive=True))
if CHECKPOINT == "selected":
    step = json.load(open(sel[0]))["selected"] if sel else sorted(CKPTS)[-1]
else:
    step = CHECKPOINT
ck = CKPTS[step]; print("driving checkpoint", RUN, step, "(selected.json found)" if sel else "(last checkpoint)")
t0 = time.time()
r = subprocess.run([sys.executable, "sim/vla-bed/dagger.py", "relabel", "--base", RECIPE, "--run", RUN, "--checkpoint", ck, "--episodes", str(EPISODES), "--output-root", "/tmp/dagger", "--sampling-seed", str(SAMPLING_SEED)], capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:] if r.returncode else "", f"\n{(time.time() - t0) / 3600:.2f} h")
assert r.returncode == 0, "dagger.py failed"
m = json.load(open("/tmp/dagger/v7/manifest.json")); s = m["splits"]["train"]
print("rollouts:", {k: s[k] for k in ("episode_count", "frame_count", "success_count", "policy_success_rate", "wall_s")}, "driven by", m["relabel"])

In [ ]:
import shutil, pathlib, hashlib
out = shutil.make_archive("/kaggle/working/vla-bed-v7", "zip", "/tmp/dagger")   # v7/manifest.json + v7/train/{data,meta,videos}
print(out, round(pathlib.Path(out).stat().st_size / 1e6, 1), "MB", "sha256", hashlib.sha256(open(out, "rb").read()).hexdigest())
print("next: train.ipynb with inputs vla-bed-<base> + THIS notebook's output (it merges base ∪ v7 before training); on the workstation: gpu/kaggle_import.sh <zip> <sha256> imports the manifest")